In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

current_dir = Path.cwd()
proj_dir = current_dir.parent.parent.resolve() # get project root directory

print(f"Project directory set to: {proj_dir}")

# Add the project root to the system path to import local modules
sys.path.append(str(proj_dir / 'python'))

Project directory set to: C:\Users\T22127A\Documents\priv\Uni\priv_dev\dermascan


In [3]:
from pipeline.pipeline_orchestrator import PipelineOrchestrator
from pipeline.dataclasses.pipeline_config import PipelineConfig, BranchConfig, ModelConfig, TrainerConfig, TrainValTestSplitRatio, AugmentationConfig, CustomSplitConfig
from pipeline.dataclasses.enums import ModelType, FilterType, DatasetLoaderType, AugmentationType, SplitType
from pipeline.dataclasses.training_configs import AdamConfig, ReduceLROnPlateauConfig

In [4]:
pipeline_config = PipelineConfig(
    pipeline_id="test_pipeline",
    data_path=str(proj_dir / "data"),
    split_config=CustomSplitConfig(
        train_csv_path=Path(proj_dir / "data" / "custom_split" / "train_df.csv"),
        val_csv_path=Path(proj_dir / "data" / "custom_split" / "val_df.csv"),
        test_csv_path=Path(proj_dir / "data" / "custom_split" / "test_df.csv"),
    ),
    branches=[
        BranchConfig(
            branch_id="first_test_branch",
            branch_model_config=ModelConfig(
                model=ModelType.RESNET_18,
                filter_type=FilterType.RGB,
                num_classes=2,
                pretrained=True,
                freeze_weights=True
            ),
            trainer_config=TrainerConfig(
                batch_size=32,
                num_epochs=2,
                learning_rate=0.001,
                optimizer=AdamConfig(),
                scheduler=ReduceLROnPlateauConfig()
            ),
            dataset_loader=DatasetLoaderType.RGB,
            augmentation=AugmentationConfig(
                train=[AugmentationType.RANDOM_HORIZONTAL_FLIP, AugmentationType.RANDOM_RESIZE_CROP],
                val=[AugmentationType.CENTER_CROP],
            )
        ),
    ],
)

In [5]:
# Save the pipeline configuration to a JSON file
with open(proj_dir / "python" / "pipeline" / "doc" / "pipeline_config.json", "w") as f:
    f.write(pipeline_config.model_dump_json(indent=2))

In [6]:
# Load the pipeline configuration from the saved JSON file
json_config = Path(proj_dir / "python" / "pipeline" / "doc" / "pipeline_config.json").read_text()
pipeline_config = PipelineConfig.model_validate_json(json_config)

In [7]:
pipeline_runner = PipelineOrchestrator(
    pipeline_config=pipeline_config, 
    project_root_path=proj_dir
    )

In [9]:
run = pipeline_runner.run()

Starting pipeline run: test_pipeline
Pipeline run path: C:\Users\T22127A\Documents\priv\Uni\priv_dev\dermascan\results\test_pipeline_2026-04-23_17-27
----------
[BranchRunner] Using device: cpu


Epochs:   0%|          | 0/2 [00:00<?, ?it/s]

Training:   0%|          | 0/184 [00:14<?, ?it/s]

Epoch 1/2 - Train Loss: 0.1149, Train Acc: 0.8574 - Val Loss: 0.1107, Val Acc: 0.8581, Val F1 Macro: 0.4618


Training:   0%|          | 0/184 [00:00<?, ?it/s]

Epoch 2/2 - Train Loss: 0.0924, Train Acc: 0.8601 - Val Loss: 0.0998, Val Acc: 0.8581, Val F1 Macro: 0.4618


In [7]:
run

[BranchResult(branch_id='first_test_branch', training_history=TrainingHistory(train_loss=[0.12791629597081564, 0.09487255454008932], val_loss=[0.10416130184212609, 0.09572485798155256], train_accuracy=[0.853382084095064, 0.8583180987202925], val_accuracy=[0.8576300085251491, 0.8576300085251491], val_f1_macro=[0.4616796695731987, 0.4616796695731987]), metrics_result=MetricsResult(f1_macro=0.4616796695731987, f1_weighted=0.791900677903901, accuracy=0.8576300085251491, precision=0.4288150042625746, sensitivity=0.0, specificity=1.0, auc=0.820960464756372, mcc=0.0))]